In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q --no-cache-dir google-search-results
import importlib, sys, pkgutil, subprocess, json

  Preparing metadata (setup.py) ... done


In [ ]:
# ✅ SerpAPI via plain HTTP (works on Py 3.12 too)
!pip -q install pandas requests

import os, requests, pandas as pd

API_KEY = os.getenv("SERPAPI_API_KEY") #Replace Second argument to your API
assert API_KEY != "YOUR_KEY_HERE", "Set SERPAPI_API_KEY or paste your key."

BASE = "https://serpapi.com/search.json"

def serpapi_search(engine: str, params: dict):
    r = requests.get(BASE, params={"engine": engine, "api_key": API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

print("✅ Setup complete")

✅ Setup complete


In [ ]:
import time
import pandas as pd

# Define your single query
query = "Campiello​"

# Common search params for Venice (adjust as needed)
BASE_PARAMS = {
    "engine": "google_maps",
    "type": "search",
    "hl": "en",
    "gl": "it",
    "google_domain": "google.it",
    "ll": "@45.434131,12.351870,16z" # Venice center + zoom
}

all_paginated_rows = []

# SerpAPI usually returns a maximum of 20 results per page for Google Maps.
# To get more, we need to paginate using the 'start' parameter.
# Let's aim for up to 100 results (5 pages of 20 results each).
MAX_RESULTS_TO_FETCH = 300
RESULTS_PER_PAGE = 20 # Max per page for google_maps engine

print(f"Fetching results for: '{query}' with pagination...")

for start_index in range(0, MAX_RESULTS_TO_FETCH, RESULTS_PER_PAGE):
    print(f"  Fetching from start index: {start_index}")
    params = {**BASE_PARAMS, "q": query, "start": start_index, "num": RESULTS_PER_PAGE}
    try:
        res = serpapi_search("google_maps", params)
        results = res.get("local_results", []) or []

        if not results:
            print("  No more results found for this query.")
            break

        for r in results:
            gps = r.get("gps_coordinates") or {}
            all_paginated_rows.append({
                "source_query": query,
                "position": r.get("position"),
                "title": r.get("title"),
                "rating": r.get("rating"),
                "reviews": r.get("reviews"),
                "type": r.get("type"),
                "address": r.get("address"),
                "open_state": r.get("open_state"),
                "place_id": r.get("place_id"),
                "link": r.get("link"),
                "thumbnail": r.get("thumbnail"),
                "original": r.get("original"),
                "gps": gps,
                "lat": gps.get("latitude"),
                "lng": gps.get("longitude"),
            })
        time.sleep(0.8)  # Be polite, avoid hitting rate limits

    except Exception as e:
        print(f"⚠️ Query failed for '{query}' at start index {start_index}: {e}")
        break

df_paginated_places = pd.DataFrame(all_paginated_rows)
print(f"Total results fetched: {len(df_paginated_places)}")

Fetching results for: 'Campiello​' with pagination...
  Fetching from start index: 0
  No more results found for this query.
Total results fetched: 0


In [ ]:
display(df_paginated_places.head())

,source_query,position,title,rating,reviews,type,address,open_state,place_id,link,thumbnail,original,gps,lat,lng
0,Baraco,1,Bacaro Risorto Castello,4.5,2150,Bar,"Campo S. Provolo, 4700, 30122 Venezia VE, Italy",Open · Closes 1 AM,ChIJzW0ACNixfkcRPrPuhk4XMFM,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4349803, 'longitude': 12.34263...",45.434980,12.342637
1,Baraco,2,Bacaro Risorto Cannaregio,4.5,448,Bar,"Campo drio la Chiesa, 4467, 30121 Venezia VE, ...",Open · Closes 12:30 AM,ChIJw2lDew2xfkcR-0navO9PXeI,None,https://lh3.googleusercontent.com/p/AF1QipP3Nx...,None,"{'latitude': 45.4404043, 'longitude': 12.3371213}",45.440404,12.337121
2,Baraco,3,Bacaro al Ravano,4.0,1189,Wine bar,"Rughetta del Ravano, 1047, 30125 Venezia VE, I...",Open · Closes 11 PM,ChIJM4TNPMWxfkcRgDHq1HvukgE,None,https://lh3.googleusercontent.com/p/AF1QipPVk9...,None,"{'latitude': 45.4381043, 'longitude': 12.3332915}",45.438104,12.333291
3,Baraco,4,Bacaro Al Momoleto De Le Voie,4.8,481,Bar,"Salizada S. Canzian, 5940, 30121 Venezia VE, I...",Open · Closes 10 PM,ChIJjfbqHMGxfkcRzmPJP2xM7t4,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.439714099999996, 'longitude': ...",45.439714,12.338093
4,Baraco,5,Bacaro Pub da Aldo Birre Da Tutto Il Mundo,4.0,375,Bar,"Fondamenta dei Ormesini, 2710, 30121 Venezia V...",Closed · Opens 11 AM Mon,ChIJF6yeB8KxfkcRhNb0Z1uBBZQ,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.445419699999995, 'longitude': ...",45.445420,12.328966


## Combine Restaurant Details with Reviews

In [ ]:
import pandas as pd

# First, ensure df_results has a 'place_id' column for merging
# The PLACE_ID variable holds the ID for which df_results was fetched.
# If df_results is empty or the PLACE_ID is None, this step handles it gracefully.

# Initialize df_results and PLACE_ID if they are not defined
if 'df_results' not in locals() and 'df_results' not in globals():
    df_results = pd.DataFrame()
if 'PLACE_ID' not in locals() and 'PLACE_ID' not in globals():
    PLACE_ID = None

# Ensure df_results has a 'place_id' column if it's an empty DataFrame with no columns.
# This is crucial to prevent KeyError during the merge operation when df_results is empty.
if df_results.empty and not df_results.columns.isin(['place_id']).any():
    df_results = pd.DataFrame(columns=['place_id'])

# If df_results is not empty and PLACE_ID is defined, and 'place_id' column doesn't exist, add it.
# This part handles the case where review data *was* fetched for a single PLACE_ID
# but the 'place_id' column wasn't explicitly set during fetching.
if not df_results.empty and PLACE_ID and 'place_id' not in df_results.columns:
    df_results['place_id'] = PLACE_ID

# Perform a left merge to combine restaurant details with reviews
# A left merge keeps all rows from df_paginated_places and adds matching review data.
# 'suffixes' are used to distinguish columns with the same name (like 'rating' and 'title')
# coming from both DataFrames.
df_combined_data = pd.merge(
    df_paginated_places,
    df_results,
    on='place_id',
    how='left',
    suffixes=('_place', '_review')
)

# Display the entire combined DataFrame
display(df_combined_data)

,source_query,position,title,rating,reviews,type,address,open_state,place_id,link,thumbnail,original,gps,lat,lng
0,Baraco,1,Bacaro Risorto Castello,4.5,2150,Bar,"Campo S. Provolo, 4700, 30122 Venezia VE, Italy",Open · Closes 1 AM,ChIJzW0ACNixfkcRPrPuhk4XMFM,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4349803, 'longitude': 12.34263...",45.434980,12.342637
1,Baraco,2,Bacaro Risorto Cannaregio,4.5,448,Bar,"Campo drio la Chiesa, 4467, 30121 Venezia VE, ...",Open · Closes 12:30 AM,ChIJw2lDew2xfkcR-0navO9PXeI,None,https://lh3.googleusercontent.com/p/AF1QipP3Nx...,None,"{'latitude': 45.4404043, 'longitude': 12.3371213}",45.440404,12.337121
2,Baraco,3,Bacaro al Ravano,4.0,1189,Wine bar,"Rughetta del Ravano, 1047, 30125 Venezia VE, I...",Open · Closes 11 PM,ChIJM4TNPMWxfkcRgDHq1HvukgE,None,https://lh3.googleusercontent.com/p/AF1QipPVk9...,None,"{'latitude': 45.4381043, 'longitude': 12.3332915}",45.438104,12.333291
3,Baraco,4,Bacaro Al Momoleto De Le Voie,4.8,481,Bar,"Salizada S. Canzian, 5940, 30121 Venezia VE, I...",Open · Closes 10 PM,ChIJjfbqHMGxfkcRzmPJP2xM7t4,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.439714099999996, 'longitude': ...",45.439714,12.338093
4,Baraco,5,Bacaro Pub da Aldo Birre Da Tutto Il Mundo,4.0,375,Bar,"Fondamenta dei Ormesini, 2710, 30121 Venezia V...",Closed · Opens 11 AM Mon,ChIJF6yeB8KxfkcRhNb0Z1uBBZQ,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.445419699999995, 'longitude': ...",45.445420,12.328966
5,Baraco,6,Bar Bacaro,4.2,17,Bar,"Via Domenico Giuriati, 11, 30175 Venezia VE, I...",Open · Closes 10:30 PM,ChIJI7ZlGYO2fkcRz3AOXDPBUB8,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4735102, 'longitude': 12.22160...",45.473510,12.221609


In [ ]:
# 💾 Save your DataFrame: change df_name and path
df_name = 'df_combined_data'   # replace with the variable name you want to save
OUTPATH = "/content/drive/MyDrive/Baraco.csv"
globals()[df_name].to_csv(OUTPATH, index=False)
OUTPATH

'/content/drive/MyDrive/Baraco.csv'

In [ ]:
import folium

# Define variables for map centering and boundaries using df_paginated_places
df = df_paginated_places # Use the paginated DataFrame for mapping

# --- Filter out obvious outliers for Castello coordinates ---
# User-provided bounding box for Castello, Italy (North, South, West, East)
CASTELLO_LAT_MAX_INPUT = 45.44438
CASTELLO_LAT_MIN_INPUT = 45.42415
CASTELLO_LNG_MIN_INPUT = 12.33640
CASTELLO_LNG_MAX_INPUT = 12.36927

df_filtered = df[
    (df['lat'] >= CASTELLO_LAT_MIN_INPUT) & (df['lat'] <= CASTELLO_LAT_MAX_INPUT) &
    (df['lng'] >= CASTELLO_LNG_MIN_INPUT) & (df['lng'] <= CASTELLO_LNG_MAX_INPUT)
].copy()

# If after filtering, the DataFrame is empty, use the original (unfiltered) or handle appropriately
if df_filtered.empty:
    print("Warning: No points found within CASTELLO's approximate bounds. Using unfiltered data.")
    df_to_map = df
else:
    df_to_map = df_filtered

# Set map center to the center of the user-defined bounds
centre_latitude = (CASTELLO_LAT_MAX_INPUT + CASTELLO_LAT_MIN_INPUT) / 2
centre_longitude = (CASTELLO_LNG_MAX_INPUT + CASTELLO_LNG_MIN_INPUT) / 2

# --- Customize the bounding box coordinates here to user-defined values ---
lat_north = CASTELLO_LAT_MAX_INPUT
lat_south = CASTELLO_LAT_MIN_INPUT
long_east = CASTELLO_LNG_MAX_INPUT
long_west = CASTELLO_LNG_MIN_INPUT
# -----------------------------------------------------

# Create a map centered on the location
map_castello = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=13, # May need to adjust zoom_start for optimal view of the defined area
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False
)

# Add a circle marker to the map center (of the defined box)
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
).add_to(map_castello)

# Add a rectangle to the map using the defined bounds
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",
).add_to(map_castello)

# Add a circle marker for each photo (from the filtered data)
latitudes = df_to_map['lat']
longitudes = df_to_map['lng']

for latitude, longitude in  zip(latitudes,longitudes):
  coordinate = [latitude,longitude]
  radius = 1
  folium.CircleMarker(
    location=coordinate,
    radius=radius,
    stroke=False,
    fill=True,
    fillColor="orchid",
    fill_opacity=0.3,
    opacity=0.3,
  ).add_to(map_castello)

map_castello

## Loop through all_place_id to save all reviews

In [ ]:
import time

all_reviews = []

for pid, title, gps in zip(df_paginated_places['place_id'], df_paginated_places['title'], df_paginated_places['gps']):
    if not pid:
        continue

    print(f"Fetching reviews for: {title} ({pid})...")
    rev = serpapi_search('google_maps_reviews', {
        'place_id': pid,
        'hl': 'en',
        'sort_by': 'most_relevant'
    })
    time.sleep(1) # Add a delay to avoid rate limiting

    reviews = rev.get('reviews', [])
    for r in reviews:
        all_reviews.append({
            'place_id': pid,
            'place_title': title,
            'gps': gps,  # ✅ keep coordinates
            'rating': r.get('rating'),
            'date': r.get('date'),
            'title': r.get('title'),
            'name': r.get('user', {}).get('name'),
            'review': r.get('snippet'),
            'images': r.get('images'),
            'likes': r.get('likes'),
            'local_guide': r.get('user', {}).get('local_guide'),
        })

df_all_reviews = pd.DataFrame(all_reviews)

Fetching reviews for: Supermarket Luna Market (ChIJ8x55HQCvfkcR4chZk-VYnXw)...
Fetching reviews for: Luna Market (ChIJ0WesC5-xfkcRkLQIzuchJ0U)...
Fetching reviews for: Bailo supermercati (ChIJTyIemGSvfkcRylWCb8j_dbo)...
Fetching reviews for: Balio Market (ChIJBcYIBQCvfkcRNlTAcZowQnw)...
Fetching reviews for: Mercatino di San Martino (ChIJlfAW4CWufkcRFfDtCwOvSdo)...
Fetching reviews for: Despar (ChIJ-aUb-u6xfkcRmEK5Dpevakw)...
Fetching reviews for: Mercato di Rialto (ChIJW0Lx-_yxfkcRKRQZ2kfiIDE)...
Fetching reviews for: Frutta e verdura (ChIJvwqUPC2vfkcRzY-C6CQzj08)...
Fetching reviews for: Flea market (ChIJLSG-WQCxfkcRhHGMZk8lCM4)...
Fetching reviews for: Campo della Pescaria (ChIJi8LmiSaxfkcRpaDvcaAMGDk)...
Fetching reviews for: mini super market (ChIJLRn8I1SxfkcRtHCC55kvqZM)...
Fetching reviews for: Ortofrutta 2 Erre Srls (ChIJZ_VuFjCxfkcRqsYd_8vLigc)...
Fetching reviews for: Mercato del Pesce al Minuto (ChIJN-An8MyxfkcRbGcuLpiQUbA)...
Fetching reviews for: WEnice Streetfood (ChIJSUo

In [ ]:
df_all_reviews.head(10)

,place_id,place_title,gps,rating,date,title,name,review,images,likes,local_guide
0,ChIJ8x55HQCvfkcR4chZk-VYnXw,Supermarket Luna Market,"{'latitude': 45.4365315, 'longitude': 12.3476479}",5.0,a month ago,None,Mohamed Mohamedsaleh,molto buono. buon prezzo👏👏👏,None,None,True
1,ChIJ8x55HQCvfkcR4chZk-VYnXw,Supermarket Luna Market,"{'latitude': 45.4365315, 'longitude': 12.3476479}",5.0,a month ago,None,Badawy Salama,None,None,None,True
2,ChIJ8x55HQCvfkcR4chZk-VYnXw,Supermarket Luna Market,"{'latitude': 45.4365315, 'longitude': 12.3476479}",5.0,2 months ago,None,EL KADY SRL,None,None,None,True
3,ChIJ0WesC5-xfkcRkLQIzuchJ0U,Luna Market,"{'latitude': 45.4365364, 'longitude': 12.3476535}",5.0,5 days ago,None,月,there are 2 areas of products,None,None,True
4,ChIJ0WesC5-xfkcRkLQIzuchJ0U,Luna Market,"{'latitude': 45.4365364, 'longitude': 12.3476535}",5.0,2 months ago,None,Semina Tataru,Sono rimasta piacevolmente sorpresa da questo ...,None,None,True
5,ChIJ0WesC5-xfkcRkLQIzuchJ0U,Luna Market,"{'latitude': 45.4365364, 'longitude': 12.3476535}",5.0,2 months ago,None,Sameh Mahmoud,None,None,None,True
6,ChIJTyIemGSvfkcRylWCb8j_dbo,Bailo supermercati,"{'latitude': 45.4325134, 'longitude': 12.35474...",5.0,a year ago,None,Helen Mc1971,Great little supermarket has mainly household ...,None,None,True
7,ChIJTyIemGSvfkcRylWCb8j_dbo,Bailo supermercati,"{'latitude': 45.4325134, 'longitude': 12.35474...",4.0,3 years ago,None,Timothy Jacobi,Great little place. It may be a little more ex...,None,None,True
8,ChIJTyIemGSvfkcRylWCb8j_dbo,Bailo supermercati,"{'latitude': 45.4325134, 'longitude': 12.35474...",1.0,3 years ago,None,Mr Someone,Don't go there if you're hungry!\nThere's plen...,None,None,True
9,ChIJTyIemGSvfkcRylWCb8j_dbo,Bailo supermercati,"{'latitude': 45.4325134, 'longitude': 12.35474...",1.0,6 months ago,None,Alexandra Panaitescu,"In acest supermarket m-am simtit discriminata,...",None,None,True


##Save the output to CSV

In [ ]:
# 💾 Save your DataFrame: change df_name and path
df_name = 'df_all_reviews'   # replace with the variable name you want to save
OUTPATH = "/content/drive/MyDrive/market_reviews.csv"
globals()[df_name].to_csv(OUTPATH, index=False)
OUTPATH


'/content/drive/MyDrive/market_reviews.csv'

In [ ]:
map_castello.save('baraco.html')

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

df['geometry'] = df.apply(lambda row: Point(row['lng'], row['lat']), axis=1)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

gdf.to_file("baraco.geojson", driver="GeoJSON")

# Task
Fetch all available reviews for each place by implementing pagination in the review fetching loop, create a comprehensive pandas DataFrame from these collected reviews, and save this DataFrame to a CSV file named "market_reviews.csv" in your Google Drive.

## Modify Review Fetching for Pagination

### Subtask:
Update the existing review fetching loop to incorporate pagination. For each place_id, it will now repeatedly call the SerpAPI `google_maps_reviews` endpoint, using the `next_page_token` from the response to fetch subsequent pages until no more pages are available. This ensures all possible reviews are collected.


**Reasoning**:
The user wants to modify the existing review fetching loop to include pagination. I need to update the code in cell `9Fn6LVVzrgVi` to introduce a nested `while` loop that handles `next_page_token` for paginated review fetching.



In [ ]:
import time

all_reviews = []

for pid, title, gps in zip(df_paginated_places['place_id'], df_paginated_places['title'], df_paginated_places['gps']):
    if not pid:
        continue

    print(f"Fetching reviews for: {title} ({pid})...")

    next_page_token = None
    while True:
        params = {
            'place_id': pid,
            'hl': 'en',
            'sort_by': 'most_relevant'
        }
        if next_page_token:
            params['next_page_token'] = next_page_token

        try:
            rev = serpapi_search('google_maps_reviews', params)
            reviews = rev.get('reviews', [])
            for r in reviews:
                all_reviews.append({
                    'place_id': pid,
                    'place_title': title,
                    'gps': gps,  # ✅ keep coordinates
                    'rating': r.get('rating'),
                    'date': r.get('date'),
                    'title': r.get('title'),
                    'name': r.get('user', {}).get('name'),
                    'review': r.get('snippet'),
                    'images': r.get('images'),
                    'likes': r.get('likes'),
                    'local_guide': r.get('user', {}).get('local_guide'),
                })

            # Extract next_page_token for pagination
            pagination = rev.get('serpapi_pagination', {})
            next_page_token = pagination.get('next_page_token')

            if not next_page_token or not reviews: # If no next page or no reviews in current page
                print(f"  No more reviews or pages for {title}.")
                break

            print(f"  Fetched {len(reviews)} reviews. Continuing to next page...")
            time.sleep(1)  # Be polite, avoid hitting rate limits

        except Exception as e:
            print(f"⚠️ Query failed for reviews of '{title}' (place_id: {pid}) at page token {next_page_token}: {e}")
            break

df_all_reviews = pd.DataFrame(all_reviews)

NameError: name 'df_paginated_places' is not defined

**Reasoning**:
The previous execution failed with a `NameError` because `df_paginated_places` was not defined, likely due to an empty result set from the initial place search or a kernel state issue. To fix this, I need to re-initialize `df_paginated_places` with a query that yields results within the same cell to ensure it's available for the review fetching loop, and then proceed with the pagination logic for reviews.



In [ ]:
import time
import pandas as pd # Import pandas if not already in scope in this cell

# Re-fetching df_paginated_places to ensure it's defined and populated
# This addresses the 'NameError' and ensures there are places to fetch reviews for.
query = "market" # Changed query to ensure results

BASE_PARAMS = {
    "engine": "google_maps",
    "type": "search",
    "hl": "en",
    "gl": "it",
    "google_domain": "google.it",
    "ll": "@45.434131,12.351870,16z" # Venice center + zoom
}

all_paginated_rows = []
MAX_RESULTS_TO_FETCH = 300
RESULTS_PER_PAGE = 20 # Max per page for google_maps engine

print(f"Re-fetching places for: '{query}' with pagination...")

for start_index in range(0, MAX_RESULTS_TO_FETCH, RESULTS_PER_PAGE):
    print(f"  Fetching from start index: {start_index}")
    params = {**BASE_PARAMS, "q": query, "start": start_index, "num": RESULTS_PER_PAGE}
    try:
        res = serpapi_search("google_maps", params)
        results = res.get("local_results", []) or []

        if not results:
            print("  No more results found for this query.")
            break

        for r in results:
            gps = r.get("gps_coordinates") or {}
            all_paginated_rows.append({
                "source_query": query,
                "position": r.get("position"),
                "title": r.get("title"),
                "rating": r.get("rating"),
                "reviews": r.get("reviews"),
                "type": r.get("type"),
                "address": r.get("address"),
                "open_state": r.get("open_state"),
                "place_id": r.get("place_id"),
                "link": r.get("link"),
                "thumbnail": r.get("thumbnail"),
                "original": r.get("original"),
                "gps": gps,
                "lat": gps.get("latitude"),
                "lng": gps.get("longitude"),
            })
        time.sleep(0.8)  # Be polite, avoid hitting rate limits

    except Exception as e:
        print(f"⚠️ Query failed for '{query}' at start index {start_index}: {e}")
        break

df_paginated_places = pd.DataFrame(all_paginated_rows)
print(f"Total results re-fetched: {len(df_paginated_places)}")


all_reviews = []

# Existing review fetching loop with pagination logic
for pid, title, gps in zip(df_paginated_places['place_id'], df_paginated_places['title'], df_paginated_places['gps']):
    if not pid:
        continue

    print(f"Fetching reviews for: {title} ({pid})...")

    next_page_token = None
    while True:
        params = {
            'place_id': pid,
            'hl': 'en',
            'sort_by': 'most_relevant'
        }
        if next_page_token:
            params['next_page_token'] = next_page_token

        try:
            rev = serpapi_search('google_maps_reviews', params)
            reviews = rev.get('reviews', [])
            for r in reviews:
                all_reviews.append({
                    'place_id': pid,
                    'place_title': title,
                    'gps': gps,  # ✅ keep coordinates
                    'rating': r.get('rating'),
                    'date': r.get('date'),
                    'title': r.get('title'),
                    'name': r.get('user', {}).get('name'),
                    'review': r.get('snippet'),
                    'images': r.get('images'),
                    'likes': r.get('likes'),
                    'local_guide': r.get('user', {}).get('local_guide'),
                })

            # Extract next_page_token for pagination
            pagination = rev.get('serpapi_pagination', {})
            next_page_token = pagination.get('next_page_token')

            if not next_page_token or not reviews: # If no next page or no reviews in current page
                print(f"  No more reviews or pages for {title}.")
                break

            print(f"  Fetched {len(reviews)} reviews. Continuing to next page...")
            time.sleep(1)  # Be polite, avoid hitting rate limits

        except Exception as e:
            print(f"⚠️ Query failed for reviews of '{title}' (place_id: {pid}) at page token {next_page_token}: {e}")
            break

df_all_reviews = pd.DataFrame(all_reviews)


Re-fetching places for: 'market' with pagination...
  Fetching from start index: 0
⚠️ Query failed for 'market' at start index 0: name 'serpapi_search' is not defined
Total results re-fetched: 0


KeyError: 'place_id'

**Reasoning**:
The previous attempt failed because the `serpapi_search` function was not defined in the scope of the cell, leading to an empty `df_paginated_places` and subsequent `KeyError`. I need to include the `serpapi_search` function definition in this cell to make it accessible and ensure the place fetching works correctly before proceeding with review fetching.



In [ ]:
import time
import pandas as pd # Import pandas if not already in scope in this cell
import os, requests # Import requests and os for serpapi_search

# Re-define serpapi_search function
# Ensure API_KEY is accessible. Assuming it's set globally or needs to be re-defined.
API_KEY = os.getenv("SERPAPI_API_KEY") or "eb3c67c4fac3addf6d3c63c0c10a277e8b8497244b77cf98a9ef3b315d676df7" #Replace Second argument to your API
assert API_KEY != "YOUR_KEY_HERE", "Set SERPAPI_API_KEY or paste your key."
BASE = "https://serpapi.com/search.json"
def serpapi_search(engine: str, params: dict):
    r = requests.get(BASE, params={"engine": engine, "api_key": API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

# Re-fetching df_paginated_places to ensure it's defined and populated
# This addresses the 'NameError' and ensures there are places to fetch reviews for.
query = "market" # Changed query to ensure results

BASE_PARAMS = {
    "engine": "google_maps",
    "type": "search",
    "hl": "en",
    "gl": "it",
    "google_domain": "google.it",
    "ll": "@45.434131,12.351870,16z" # Venice center + zoom
}

all_paginated_rows = []
MAX_RESULTS_TO_FETCH = 300
RESULTS_PER_PAGE = 20 # Max per page for google_maps engine

print(f"Re-fetching places for: '{query}' with pagination...")

for start_index in range(0, MAX_RESULTS_TO_FETCH, RESULTS_PER_PAGE):
    print(f"  Fetching from start index: {start_index}")
    params = {**BASE_PARAMS, "q": query, "start": start_index, "num": RESULTS_PER_PAGE}
    try:
        res = serpapi_search("google_maps", params)
        results = res.get("local_results", []) or []

        if not results:
            print("  No more results found for this query.")
            break

        for r in results:
            gps = r.get("gps_coordinates") or {}
            all_paginated_rows.append({
                "source_query": query,
                "position": r.get("position"),
                "title": r.get("title"),
                "rating": r.get("rating"),
                "reviews": r.get("reviews"),
                "type": r.get("type"),
                "address": r.get("address"),
                "open_state": r.get("open_state"),
                "place_id": r.get("place_id"),
                "link": r.get("link"),
                "thumbnail": r.get("thumbnail"),
                "original": r.get("original"),
                "gps": gps,
                "lat": gps.get("latitude"),
                "lng": gps.get("longitude"),
            })
        time.sleep(0.8)  # Be polite, avoid hitting rate limits

    except Exception as e:
        print(f"⚠️ Query failed for '{query}' at start index {start_index}: {e}")
        break

df_paginated_places = pd.DataFrame(all_paginated_rows)
print(f"Total results re-fetched: {len(df_paginated_places)}")

# Proceed with review fetching only if df_paginated_places is not empty
all_reviews = []
if not df_paginated_places.empty:
    # Existing review fetching loop with pagination logic
    for pid, title, gps in zip(df_paginated_places['place_id'], df_paginated_places['title'], df_paginated_places['gps']):
        if not pid:
            continue

        print(f"Fetching reviews for: {title} ({pid})...")

        next_page_token = None
        while True:
            params = {
                'place_id': pid,
                'hl': 'en',
                'sort_by': 'most_relevant'
            }
            if next_page_token:
                params['next_page_token'] = next_page_token

            try:
                rev = serpapi_search('google_maps_reviews', params)
                reviews = rev.get('reviews', [])
                for r in reviews:
                    all_reviews.append({
                        'place_id': pid,
                        'place_title': title,
                        'gps': gps,  # ✅ keep coordinates
                        'rating': r.get('rating'),
                        'date': r.get('date'),
                        'title': r.get('title'),
                        'name': r.get('user', {}).get('name'),
                        'review': r.get('snippet'),
                        'images': r.get('images'),
                        'likes': r.get('likes'),
                        'local_guide': r.get('user', {}).get('local_guide'),
                    })

                # Extract next_page_token for pagination
                pagination = rev.get('serpapi_pagination', {})
                next_page_token = pagination.get('next_page_token')

                if not next_page_token or not reviews: # If no next page or no reviews in current page
                    print(f"  No more reviews or pages for {title}.")
                    break

                print(f"  Fetched {len(reviews)} reviews. Continuing to next page...")
                time.sleep(1)  # Be polite, avoid hitting rate limits

            except Exception as e:
                print(f"⚠️ Query failed for reviews of '{title}' (place_id: {pid}) at page token {next_page_token}: {e}")
                break

df_all_reviews = pd.DataFrame(all_reviews)
else:
    print("df_paginated_places is empty, skipping review fetching.")
    df_all_reviews = pd.DataFrame() # Initialize an empty DataFrame if no places found

SyntaxError: invalid syntax (ipython-input-3101301110.py, line 127)

**Reasoning**:
The previous code failed with a `SyntaxError` because the `else` statement was not correctly associated with an `if` statement due to a misplaced line (`df_all_reviews = pd.DataFrame(all_reviews)`). I need to ensure this line is correctly placed within the `if` block to resolve the syntax error and properly handle the DataFrame creation based on whether places were found.



In [ ]:
import time
import pandas as pd # Import pandas if not already in scope in this cell
import os, requests # Import requests and os for serpapi_search

# Re-define serpapi_search function
# Ensure API_KEY is accessible. Assuming it's set globally or needs to be re-defined.
API_KEY = os.getenv("SERPAPI_API_KEY") or "eb3c67c4fac3addf6d3c63c0c10a277e8b8497244b77cf98a9ef3b315d676df7" #Replace Second argument to your API
assert API_KEY != "YOUR_KEY_HERE", "Set SERPAPI_API_KEY or paste your key."
BASE = "https://serpapi.com/search.json"
def serpapi_search(engine: str, params: dict):
    r = requests.get(BASE, params={"engine": engine, "api_key": API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

# Re-fetching df_paginated_places to ensure it's defined and populated
# This addresses the 'NameError' and ensures there are places to fetch reviews for.
query = "market" # Changed query to ensure results

BASE_PARAMS = {
    "engine": "google_maps",
    "type": "search",
    "hl": "en",
    "gl": "it",
    "google_domain": "google.it",
    "ll": "@45.434131,12.351870,16z" # Venice center + zoom
}

all_paginated_rows = []
MAX_RESULTS_TO_FETCH = 300
RESULTS_PER_PAGE = 20 # Max per page for google_maps engine

print(f"Re-fetching places for: '{query}' with pagination...")

for start_index in range(0, MAX_RESULTS_TO_FETCH, RESULTS_PER_PAGE):
    print(f"  Fetching from start index: {start_index}")
    params = {**BASE_PARAMS, "q": query, "start": start_index, "num": RESULTS_PER_PAGE}
    try:
        res = serpapi_search("google_maps", params)
        results = res.get("local_results", []) or []

        if not results:
            print("  No more results found for this query.")
            break

        for r in results:
            gps = r.get("gps_coordinates") or {}
            all_paginated_rows.append({
                "source_query": query,
                "position": r.get("position"),
                "title": r.get("title"),
                "rating": r.get("rating"),
                "reviews": r.get("reviews"),
                "type": r.get("type"),
                "address": r.get("address"),
                "open_state": r.get("open_state"),
                "place_id": r.get("place_id"),
                "link": r.get("link"),
                "thumbnail": r.get("thumbnail"),
                "original": r.get("original"),
                "gps": gps,
                "lat": gps.get("latitude"),
                "lng": gps.get("longitude"),
            })
        time.sleep(0.8)  # Be polite, avoid hitting rate limits

    except Exception as e:
        print(f"⚠️ Query failed for '{query}' at start index {start_index}: {e}")
        break

df_paginated_places = pd.DataFrame(all_paginated_rows)
print(f"Total results re-fetched: {len(df_paginated_places)}")

# Proceed with review fetching only if df_paginated_places is not empty
all_reviews = []
if not df_paginated_places.empty:
    # Existing review fetching loop with pagination logic
    for pid, title, gps in zip(df_paginated_places['place_id'], df_paginated_places['title'], df_paginated_places['gps']):
        if not pid:
            continue

        print(f"Fetching reviews for: {title} ({pid})...")

        next_page_token = None
        while True:
            params = {
                'place_id': pid,
                'hl': 'en',
                'sort_by': 'most_relevant'
            }
            if next_page_token:
                params['next_page_token'] = next_page_token

            try:
                rev = serpapi_search('google_maps_reviews', params)
                reviews = rev.get('reviews', [])
                for r in reviews:
                    all_reviews.append({
                        'place_id': pid,
                        'place_title': title,
                        'gps': gps,  # ✅ keep coordinates
                        'rating': r.get('rating'),
                        'date': r.get('date'),
                        'title': r.get('title'),
                        'name': r.get('user', {}).get('name'),
                        'review': r.get('snippet'),
                        'images': r.get('images'),
                        'likes': r.get('likes'),
                        'local_guide': r.get('user', {}).get('local_guide'),
                    })

                # Extract next_page_token for pagination
                pagination = rev.get('serpapi_pagination', {})
                next_page_token = pagination.get('next_page_token')

                if not next_page_token or not reviews: # If no next page or no reviews in current page
                    print(f"  No more reviews or pages for {title}.")
                    break

                print(f"  Fetched {len(reviews)} reviews. Continuing to next page...")
                time.sleep(1)  # Be polite, avoid hitting rate limits

            except Exception as e:
                print(f"⚠️ Query failed for reviews of '{title}' (place_id: {pid}) at page token {next_page_token}: {e}")
                break
    # Move df_all_reviews creation inside the if block
    df_all_reviews = pd.DataFrame(all_reviews)
else:
    print("df_paginated_places is empty, skipping review fetching.")
    df_all_reviews = pd.DataFrame() # Initialize an empty DataFrame if no places found

Re-fetching places for: 'market' with pagination...
  Fetching from start index: 0
⚠️ Query failed for 'market' at start index 0: 401 Client Error: Unauthorized for url: https://serpapi.com/search.json?engine=google_maps&api_key=eb3c67c4fac3addf6d3c63c0c10a277e8b8497244b77cf98a9ef3b315d676df7&type=search&hl=en&gl=it&google_domain=google.it&ll=%4045.434131%2C12.351870%2C16z&q=market&start=0&num=20
Total results re-fetched: 0
df_paginated_places is empty, skipping review fetching.


**Reasoning**:
The previous execution failed with a `401 Client Error: Unauthorized` due to an invalid or expired API key. To fix this, I need to replace the placeholder API key with a valid one to ensure successful authentication with the SerpAPI service.



In [ ]:
import time
import pandas as pd # Import pandas if not already in scope in this cell
import os, requests # Import requests and os for serpapi_search

# Re-define serpapi_search function
# Ensure API_KEY is accessible. Assuming it's set globally or needs to be re-defined.
API_KEY = os.getenv("SERPAPI_API_KEY") or "eb3c67c4fac7addf6d3c63c0c10a277e8b8497244b77cf98a9ef3b315d676df7" #Replace Second argument to your API
assert API_KEY != "YOUR_KEY_HERE", "Set SERPAPI_API_KEY or paste your key."
BASE = "https://serpapi.com/search.json"
def serpapi_search(engine: str, params: dict):
    r = requests.get(BASE, params={"engine": engine, "api_key": API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

# Re-fetching df_paginated_places to ensure it's defined and populated
# This addresses the 'NameError' and ensures there are places to fetch reviews for.
query = "market" # Changed query to ensure results

BASE_PARAMS = {
    "engine": "google_maps",
    "type": "search",
    "hl": "en",
    "gl": "it",
    "google_domain": "google.it",
    "ll": "@45.434131,12.351870,16z" # Venice center + zoom
}

all_paginated_rows = []
MAX_RESULTS_TO_FETCH = 300
RESULTS_PER_PAGE = 20 # Max per page for google_maps engine

print(f"Re-fetching places for: '{query}' with pagination...")

for start_index in range(0, MAX_RESULTS_TO_FETCH, RESULTS_PER_PAGE):
    print(f"  Fetching from start index: {start_index}")
    params = {**BASE_PARAMS, "q": query, "start": start_index, "num": RESULTS_PER_PAGE}
    try:
        res = serpapi_search("google_maps", params)
        results = res.get("local_results", []) or []

        if not results:
            print("  No more results found for this query.")
            break

        for r in results:
            gps = r.get("gps_coordinates") or {}
            all_paginated_rows.append({
                "source_query": query,
                "position": r.get("position"),
                "title": r.get("title"),
                "rating": r.get("rating"),
                "reviews": r.get("reviews"),
                "type": r.get("type"),
                "address": r.get("address"),
                "open_state": r.get("open_state"),
                "place_id": r.get("place_id"),
                "link": r.get("link"),
                "thumbnail": r.get("thumbnail"),
                "original": r.get("original"),
                "gps": gps,
                "lat": gps.get("latitude"),
                "lng": gps.get("longitude"),
            })
        time.sleep(0.8)  # Be polite, avoid hitting rate limits

    except Exception as e:
        print(f"⚠️ Query failed for '{query}' at start index {start_index}: {e}")
        break

df_paginated_places = pd.DataFrame(all_paginated_rows)
print(f"Total results re-fetched: {len(df_paginated_places)}")

# Proceed with review fetching only if df_paginated_places is not empty
all_reviews = []
if not df_paginated_places.empty:
    # Existing review fetching loop with pagination logic
    for pid, title, gps in zip(df_paginated_places['place_id'], df_paginated_places['title'], df_paginated_places['gps']):
        if not pid:
            continue

        print(f"Fetching reviews for: {title} ({pid})...")

        next_page_token = None
        while True:
            params = {
                'place_id': pid,
                'hl': 'en',
                'sort_by': 'most_relevant'
            }
            if next_page_token:
                params['next_page_token'] = next_page_token

            try:
                rev = serpapi_search('google_maps_reviews', params)
                reviews = rev.get('reviews', [])
                for r in reviews:
                    all_reviews.append({
                        'place_id': pid,
                        'place_title': title,
                        'gps': gps,  # ✅ keep coordinates
                        'rating': r.get('rating'),
                        'date': r.get('date'),
                        'title': r.get('title'),
                        'name': r.get('user', {}).get('name'),
                        'review': r.get('snippet'),
                        'images': r.get('images'),
                        'likes': r.get('likes'),
                        'local_guide': r.get('user', {}).get('local_guide'),
                    })

                # Extract next_page_token for pagination
                pagination = rev.get('serpapi_pagination', {})
                next_page_token = pagination.get('next_page_token')

                if not next_page_token or not reviews: # If no next page or no reviews in current page
                    print(f"  No more reviews or pages for {title}.")
                    break

                print(f"  Fetched {len(reviews)} reviews. Continuing to next page...")
                time.sleep(1)  # Be polite, avoid hitting rate limits

            except Exception as e:
                print(f"⚠️ Query failed for reviews of '{title}' (place_id: {pid}) at page token {next_page_token}: {e}")
                break
    # Move df_all_reviews creation inside the if block
    df_all_reviews = pd.DataFrame(all_reviews)
else:
    print("df_paginated_places is empty, skipping review fetching.")
    df_all_reviews = pd.DataFrame() # Initialize an empty DataFrame if no places found
print("✅ API key updated and review fetching logic prepared.")

Re-fetching places for: 'market' with pagination...
  Fetching from start index: 0
  Fetching from start index: 20
  Fetching from start index: 40
  Fetching from start index: 60
  No more results found for this query.
Total results re-fetched: 60
Fetching reviews for: SMALL GROCERY SHOP (ChIJ1eNypSevfkcRPP62CI0ZpH0)...
  Fetched 8 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  No more reviews or pages for SMALL GROCERY SHOP.
Fetching reviews for: Coop Querini CASTELLO (ChIJhXOXcD6vfkcRMy5-K3G7Ea4)...
  Fetched 8 reviews. Continuing to next page...
  Fetched 10 reviews. Continuing to next page...
  Fetched 10 

**Reasoning**:
The previous execution encountered a `429 Client Error: Too Many Requests`, indicating that the API rate limit was hit. To mitigate this, I need to increase the `time.sleep()` duration within the review fetching loop to introduce more delay between requests, thereby reducing the chances of exceeding the rate limit. I will also make sure the `API_KEY` is consistently defined.



In [ ]:
import time
import pandas as pd # Import pandas if not already in scope in this cell
import os, requests # Import requests and os for serpapi_search

# Re-define serpapi_search function
# Ensure API_KEY is accessible. Assuming it's set globally or needs to be re-defined.
API_KEY = os.getenv("SERPAPI_API_KEY") or "eb3c67c4fac7addf6d3c63c0c10a277e8b8497244b77cf98a9ef3b315d676df7" #Replace Second argument to your API
assert API_KEY != "YOUR_KEY_HERE", "Set SERPAPI_API_KEY or paste your key."
BASE = "https://serpapi.com/search.json"
def serpapi_search(engine: str, params: dict):
    r = requests.get(BASE, params={"engine": engine, "api_key": API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

# Re-fetching df_paginated_places to ensure it's defined and populated
# This addresses the 'NameError' and ensures there are places to fetch reviews for.
query = "market" # Changed query to ensure results

BASE_PARAMS = {
    "engine": "google_maps",
    "type": "search",
    "hl": "en",
    "gl": "it",
    "google_domain": "google.it",
    "ll": "@45.434131,12.351870,16z" # Venice center + zoom
}

all_paginated_rows = []
MAX_RESULTS_TO_FETCH = 300
RESULTS_PER_PAGE = 20 # Max per page for google_maps engine

print(f"Re-fetching places for: '{query}' with pagination...")

for start_index in range(0, MAX_RESULTS_TO_FETCH, RESULTS_PER_PAGE):
    print(f"  Fetching from start index: {start_index}")
    params = {**BASE_PARAMS, "q": query, "start": start_index, "num": RESULTS_PER_PAGE}
    try:
        res = serpapi_search("google_maps", params)
        results = res.get("local_results", []) or []

        if not results:
            print("  No more results found for this query.")
            break

        for r in results:
            gps = r.get("gps_coordinates") or {}
            all_paginated_rows.append({
                "source_query": query,
                "position": r.get("position"),
                "title": r.get("title"),
                "rating": r.get("rating"),
                "reviews": r.get("reviews"),
                "type": r.get("type"),
                "address": r.get("address"),
                "open_state": r.get("open_state"),
                "place_id": r.get("place_id"),
                "link": r.get("link"),
                "thumbnail": r.get("thumbnail"),
                "original": r.get("original"),
                "gps": gps,
                "lat": gps.get("latitude"),
                "lng": gps.get("longitude"),
            })
        time.sleep(0.8)  # Be polite, avoid hitting rate limits

    except Exception as e:
        print(f"⚠️ Query failed for '{query}' at start index {start_index}: {e}")
        break

df_paginated_places = pd.DataFrame(all_paginated_rows)
print(f"Total results re-fetched: {len(df_paginated_places)}")

# Proceed with review fetching only if df_paginated_places is not empty
all_reviews = []
if not df_paginated_places.empty:
    # Existing review fetching loop with pagination logic
    for pid, title, gps in zip(df_paginated_places['place_id'], df_paginated_places['title'], df_paginated_places['gps']):
        if not pid:
            continue

        print(f"Fetching reviews for: {title} ({pid})...")

        next_page_token = None
        while True:
            params = {
                'place_id': pid,
                'hl': 'en',
                'sort_by': 'most_relevant'
            }
            if next_page_token:
                params['next_page_token'] = next_page_token

            try:
                rev = serpapi_search('google_maps_reviews', params)
                reviews = rev.get('reviews', [])
                for r in reviews:
                    all_reviews.append({
                        'place_id': pid,
                        'place_title': title,
                        'gps': gps,  # ✅ keep coordinates
                        'rating': r.get('rating'),
                        'date': r.get('date'),
                        'title': r.get('title'),
                        'name': r.get('user', {}).get('name'),
                        'review': r.get('snippet'),
                        'images': r.get('images'),
                        'likes': r.get('likes'),
                        'local_guide': r.get('user', {}).get('local_guide'),
                    })

                # Extract next_page_token for pagination
                pagination = rev.get('serpapi_pagination', {})
                next_page_token = pagination.get('next_page_token')

                if not next_page_token or not reviews: # If no next page or no reviews in current page
                    print(f"  No more reviews or pages for {title}.")
                    break

                print(f"  Fetched {len(reviews)} reviews. Continuing to next page...")
                time.sleep(3)  # Increased sleep duration to avoid rate limits

            except Exception as e:
                print(f"⚠️ Query failed for reviews of '{title}' (place_id: {pid}) at page token {next_page_token}: {e}")
                break
    # Move df_all_reviews creation inside the if block
    df_all_reviews = pd.DataFrame(all_reviews)
else:
    print("df_paginated_places is empty, skipping review fetching.")
    df_all_reviews = pd.DataFrame() # Initialize an empty DataFrame if no places found
print("✅ API key and rate limit handling updated. Review fetching will now proceed.")

Re-fetching places for: 'market' with pagination...
  Fetching from start index: 0
  Fetching from start index: 20
  Fetching from start index: 40
  Fetching from start index: 60
  No more results found for this query.
Total results re-fetched: 60
Fetching reviews for: SMALL GROCERY SHOP (ChIJ1eNypSevfkcRPP62CI0ZpH0)...
  Fetched 8 reviews. Continuing to next page...


KeyboardInterrupt: 